# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tannusaini2110-spec/Internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip -q install duckdb huggingface_hub

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLE = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

feat = con.sql(f"""
    SELECT content_hash_id,
           AVG(gsc_avg_position) as avg_position,
           SUM(gsc_impressions) as total_impressions,
           SUM(gsc_clicks) as total_clicks,
           COUNT(*) as days_seen
    FROM {TABLE}
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df().fillna(0)
feat["ctr"] = feat["total_clicks"] / feat["total_impressions"].replace(0, 1)

print("Distributions of key fields:\n")
for col in ["avg_position", "total_impressions", "total_clicks", "ctr", "days_seen"]:
    print(f"{col}:")
    print(feat[col].describe())
    print()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Distributions of key fields:

avg_position:
count    176738.000000
mean         15.999277
std          17.686260
min           0.000000
25%           5.001970
50%           8.505296
75%          20.369190
max         309.000000
Name: avg_position, dtype: float64

total_impressions:
count    176738.000000
mean       1587.986675
std        5431.337724
min           1.000000
25%          20.000000
50%         173.000000
75%        1039.000000
max      617124.000000
Name: total_impressions, dtype: float64

total_clicks:
count    176738.000000
mean          4.650002
std          26.722649
min           0.000000
25%           0.000000
50%           0.000000
75%           2.000000
max        5668.000000
Name: total_clicks, dtype: float64

ctr:
count    176738.000000
mean          0.004594
std           0.037760
min           0.000000
25%           0.000000
50%           0.000000
75%           0.002158
max           1.000000
Name: ctr, dtype: float64

days_seen:
count    176738.000000
mean    

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [3]:
# Signal 1: Position vs CTR
feat["good_position"] = feat["avg_position"] <= feat["avg_position"].median()
b1 = feat.groupby("good_position")["ctr"].agg(["mean", "count"])
print("Signal 1 -- Good position (top half) vs CTR:")
print(b1)
print("Verdict: CONFIRMED -- better position clearly correlates with higher CTR.\n")

# Signal 2: Volume (impressions) vs CTR
feat["high_volume"] = feat["total_impressions"] >= feat["total_impressions"].median()
b2 = feat.groupby("high_volume")["ctr"].agg(["mean", "count"])
print("Signal 2 -- High impression volume vs CTR:")
print(b2)
print("Verdict: OPPOSITE -- high-volume pages actually have LOWER CTR (0.0027)")
print("than low-volume pages (0.0065). Broad, high-impression queries likely")
print("have lower click intent than narrower, lower-volume searches. This")
print("would have been a bad signal for a naive 'quick-win' rule.\n")

# Signal 3: Activity (days_seen) vs total_clicks
feat["high_activity"] = feat["days_seen"] >= feat["days_seen"].median()
b3 = feat.groupby("high_activity")["total_clicks"].agg(["mean", "count"])
print("Signal 3 -- High activity days vs total clicks:")
print(b3)
print("Verdict: CONFIRMED, but with a caveat -- high-activity pages show far")
print("more total clicks (8.42 vs 0.68), but total_clicks is a SUM over days,")
print("so more days-seen mechanically produces more accumulated clicks. This")
print("is partly definitional, not purely a new causal insight.\n")

Signal 1 -- Good position (top half) vs CTR:
                   mean  count
good_position                 
False          0.002624  88369
True           0.006564  88369
Verdict: CONFIRMED -- better position clearly correlates with higher CTR.

Signal 2 -- High impression volume vs CTR:
                 mean  count
high_volume                 
False        0.006532  88260
True         0.002661  88478
Verdict: OPPOSITE -- high-volume pages actually have LOWER CTR (0.0027)
than low-volume pages (0.0065). Broad, high-impression queries likely
have lower click intent than narrower, lower-volume searches. This
would have been a bad signal for a naive 'quick-win' rule.

Signal 3 -- High activity days vs total clicks:
                   mean  count
high_activity                 
False          0.675477  86077
True           8.423567  90661
Verdict: CONFIRMED, but with a caveat -- high-activity pages show far
more total clicks (8.42 vs 0.68), but total_clicks is a SUM over days,
so more days-se

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [5]:
print("Flag-linked test: Position vs CTR, linked to FlyRank's CTR-fix flag.")
print()
print("The CTR-fix flag assumes: pages ranking well but underperforming on CTR")
print("are worth a title/meta-description rewrite to capture more clicks.")
print()
print("Does the data support this assumption? Signal 1 confirms position and")
print("CTR are genuinely related (0.0066 vs 0.0026, more than double) -- so the")
print("flag's core assumption (position drives click opportunity) holds.")
print()
print("But: I should check whether GOOD position with LOW CTR specifically")
print("(the exact case the flag targets) is common enough to matter. Since CTR's")
print("median is exactly 0 (most pages get zero clicks), I use CTR == 0 with")
print("nonzero impressions as the 'underperforming' definition instead:")

flag_candidates = feat[(feat["good_position"]) & (feat["ctr"] == 0) & (feat["total_impressions"] > 0)]
print(f"\nPages with good position AND zero CTR despite impressions: {len(flag_candidates):,} "
      f"({len(flag_candidates)/len(feat):.1%} of all pages)")
print()

if len(flag_candidates) / len(feat) > 0.05:
    verdict = "CONFIRMED"
    note = "a meaningful share of pages fit this exact pattern, supporting that\nthe CTR-fix flag targets a real, sizeable opportunity."
else:
    verdict = "MIXED"
    note = "this exact pattern is rarer than expected -- the flag may still be\nuseful, but on a smaller opportunity set than assumed."

print(f"Verdict: {verdict} -- {note}")

Flag-linked test: Position vs CTR, linked to FlyRank's CTR-fix flag.

The CTR-fix flag assumes: pages ranking well but underperforming on CTR
are worth a title/meta-description rewrite to capture more clicks.

Does the data support this assumption? Signal 1 confirms position and
CTR are genuinely related (0.0066 vs 0.0026, more than double) -- so the
flag's core assumption (position drives click opportunity) holds.

But: I should check whether GOOD position with LOW CTR specifically
(the exact case the flag targets) is common enough to matter. Since CTR's
median is exactly 0 (most pages get zero clicks), I use CTR == 0 with
nonzero impressions as the 'underperforming' definition instead:

Pages with good position AND zero CTR despite impressions: 47,669 (27.0% of all pages)

Verdict: CONFIRMED -- a meaningful share of pages fit this exact pattern, supporting that
the CTR-fix flag targets a real, sizeable opportunity.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("What a content team should take from this audit:")
print()
print("1. The CTR-fix flag is well-supported: 27% of pages rank decently but")
print("   get zero clicks despite real impressions -- a large, actionable")
print("   opportunity for title/meta-description improvements.")
print()
print("2. Beware naive 'quick-win by volume' rules: high-impression pages")
print("   actually have LOWER CTR on average (Signal 2, OPPOSITE). Chasing")
print("   volume alone would misdirect effort toward already-saturated,")
print("   lower-intent pages instead of genuine underperformers.")

What a content team should take from this audit:

1. The CTR-fix flag is well-supported: 27% of pages rank decently but
   get zero clicks despite real impressions -- a large, actionable
   opportunity for title/meta-description improvements.

2. Beware naive 'quick-win by volume' rules: high-impression pages
   actually have LOWER CTR on average (Signal 2, OPPOSITE). Chasing
   volume alone would misdirect effort toward already-saturated,
   lower-intent pages instead of genuine underperformers.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.